# NILM Model Training on Google Colab

Train a Non-Intrusive Load Monitoring (NILM) disaggregation model using the
`nilm-research` package. This notebook covers the full workflow: training,
evaluation, ONNX export, and downloading artefacts.

**Runtime:** Select *Runtime > Change runtime type > T4 GPU* before running.

## 1. Install

In [ ]:
!pip install -q "nilm-research @ git+https://github.com/underscoreHQ/nilm-research.git"

## 2. GPU check

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("No GPU detected — training will be slow.")
    print("Go to Runtime > Change runtime type > T4 GPU")

## 3. Train

In [ ]:
from nilm_research.training.trainer import train_model

model, trainer = train_model(
    model_type="atcn",
    window_size=60,
    batch_size=256,
    max_epochs=30,
    learning_rate=1e-3,
    use_amda=True,
    amda_scale=2.5,
    num_workers=2,
    output_dir="outputs",
    use_wandb=False,
)

## 4. Evaluate

In [ ]:
from nilm_research.data.datamodule import NilmDataModule
from nilm_research.evaluation.evaluate import APPLIANCE_NAMES, evaluate_model

# Set up a datamodule for evaluation (no augmentation)
eval_dm = NilmDataModule(
    window_size=60,
    batch_size=256,
    num_workers=2,
    use_amda=False,
)
eval_dm.setup()

device = "cuda" if torch.cuda.is_available() else "cpu"
results = evaluate_model(model, eval_dm, device=device)

print(f"{'Metric':<12} {'Value':>10}")
print("-" * 24)
for key in ["mae", "mse", "rmse", "r2", "nde"]:
    print(f"{key:<12} {results[key]:>10.4f}")

print(f"\n{'Appliance':<14} {'MAE':>10} {'R2':>10}")
print("-" * 36)
for name in APPLIANCE_NAMES:
    print(f"{name:<14} {results[f'{name}_mae']:>10.4f} {results[f'{name}_r2']:>10.4f}")

## 5. Export to ONNX

In [ ]:
from nilm_research.export.onnx_exporter import OnnxExporter

exporter = OnnxExporter(model, output_dir="exports", model_name="nilm_colab")
normalisation_metadata = eval_dm.get_normalisation_metadata()
onnx_path = exporter.export(normalisation_metadata=normalisation_metadata)
print(f"Exported ONNX model to {onnx_path}")

## 6. Download artefacts

In [ ]:
from google.colab import files

files.download(str(onnx_path))
files.download("exports/nilm_colab_metadata.json")